# Silver Layer — Procedure Events (`procedureevents`)

**Source -> Target:** `<source_schema>.procedureevents` -> `<target_schema>.procedureevents` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Module:** this is the MIMIC-IV **ICU** (`icu.procedureevents`, MetaVision) table, not a hospital-module table. It links to `patients` on `subject_id`, `admissions` on `hadm_id`, `icustays` on `stay_id` and `d_items` on `itemid`. Presence of a row indicates a documented procedure; absence does not indicate it was not performed.

**Objective:** Standardize procedure event records while preserving patient and admission identifiers, ICU stay identifiers, procedure identifiers, timestamps, status information, and clinical measurements.

**Expected grain:** One source procedure event record. The official MIMIC-IV documentation lists no `procedureevent_id` column and does not declare a primary key; `orderid` is the candidate identifier (repeated procedures under one order carry the earlier `orderid` in `linkorderid`). Its uniqueness is reported for profiling only and is not asserted.

**Note (moved from a mislabelled header in `01_d_icd_diagnoses`):** an earlier draft header titled "Hospital Procedure Events" for this table stated the same grain idea (`orderid` where uniqueness is confirmed by the source) and the same preservation principles as above. The only additional intent kept here: procedure duration, resource utilization and operational analytics are deferred to Gold (Silver keeps `value`/`valueuom`, `starttime` and `endtime` as recorded).

**Silver rules:** Preserve every Bronze record and its clinical meaning. Standardize data types and whitespace, validate identifiers and timestamps, inspect duplicates, and reconcile Bronze–Silver counts. No deletion, filtering, deduplication, imputation or aggregation.

**Duplicate handling:** Exact duplicate rows (all columns identical) are counted and reported in one aggregation pass, but they never block publication and are never removed. The official documentation defines no unique key for this table, so identical rows may be legitimate repeated source records. No row-level duplicate flag is added: duplicates are not defined by the documentation and no profiling evidence supports a flag definition. Profile the reported counts and revisit only if evidence shows a specific, documented duplicate pattern.

**DQ flag `dq_invalid_procedure_time`:** one boolean column appended as the last column. It is `true` only when `starttime` and `endtime` are both non-null and `endtime` < `starttime`; otherwise `false` (never NULL). This is a deterministic definitional check (an interval cannot end before it starts), the same approach as `dq_invalid_transfer_time` in transfers. It is **not** backed by recorded profiling counts, and no expected count is asserted. No timestamp is corrected, and all source column values are untouched. Reversed intervals are preserved and never block publication.

**Validation policy:** Row preservation, the ordered schema and the flag being boolean and non-null are asserted unconditionally. The official `procedureevents` documentation lists no NOT NULL columns, so NULL identifiers are reported for review but never block publication.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "procedureevents"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

In [0]:
# Load and inspect ICU procedure event records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

bronze_count = bronze_df.count()
print("Bronze rows:", bronze_count)

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- location: string (nullable = true)
 |-- locationcategory: string (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- linkorderid: integer (nullable = true)
 |-- ordercategoryname: string (nullable = true)
 |-- ordercategorydescription: string (nullable = true)
 |-- patientweight: double (nullable = true)
 |-- isopenbag: integer (nullable = true)
 |-- continueinnextdept: integer (nullable = true)
 |-- statusdescription: string (nullable = true)
 |-- originalamount: double (nullable = true)
 |-- originalrate: integer (nullable = true)



subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,value,valueuom,location,locationcategory,orderid,linkorderid,ordercategoryname,ordercategorydescription,patientweight,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
10000032,29079034,39553978,18704,2180-07-23T14:43:00.000Z,2180-07-23T14:44:00.000Z,2180-07-23T14:43:00.000Z,225966,1.0,None,null,null,3050203,3050203,Procedures,Task,39.4,0,0,FinishedRunning,1.0,0
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224275,566.0,min,null,null,769190,769190,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224277,566.0,min,null,null,2668055,2668055,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1
10000690,25860671,37081114,17393,2150-11-02T20:00:00.000Z,2150-11-03T13:00:00.000Z,2150-11-03T14:50:00.000Z,224275,1020.0,min,RL Ant Forearm Medial,Peripheral - old,301419,301419,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,1020.0,1
10000690,25860671,37081114,17393,2150-11-02T20:38:00.000Z,2150-11-03T11:59:00.000Z,2150-11-03T12:34:00.000Z,224277,921.0,min,L Antecube,Peripheral - old,5207974,5207974,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,921.0,1
10000690,25860671,37081114,17393,2150-11-03T04:00:00.000Z,2150-11-03T04:01:00.000Z,2150-11-03T04:33:00.000Z,225459,1.0,None,null,null,8832126,8832126,Imaging,Task,55.3,0,0,FinishedRunning,1.0,0
10000690,25860671,37081114,33125,2150-11-05T12:00:00.000Z,2150-11-05T12:01:00.000Z,2150-11-05T12:58:00.000Z,225402,1.0,None,null,null,4411905,4411905,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225401,1.0,None,null,null,7931107,7931107,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225814,1.0,None,null,null,2290746,2290746,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0
10000690,25860671,37081114,null,2150-11-03T14:00:00.000Z,2150-11-06T17:03:00.000Z,2150-11-06T17:03:19.813Z,224277,4503.0,min,null,null,4940037,4940037,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,4503.0,1


Bronze rows: 808706


In [0]:
# Profile source row counts and NULL values across all columns
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in bronze_df.columns
    ]
)

display(bronze_profile)

total_rows,null_subject_id,null_hadm_id,null_stay_id,null_caregiver_id,null_starttime,null_endtime,null_storetime,null_itemid,null_value,null_valueuom,null_location,null_locationcategory,null_orderid,null_linkorderid,null_ordercategoryname,null_ordercategorydescription,null_patientweight,null_isopenbag,null_continueinnextdept,null_statusdescription,null_originalamount,null_originalrate
808706,0,0,0,157542,0,0,0,0,0,0,631364,631364,0,0,0,0,0,0,0,0,0,0


In [0]:
# Preserve all source fields while standardizing known data types
silver_df = bronze_df

long_columns = [
    "subject_id", "hadm_id", "stay_id",
    "itemid", "orderid", "linkorderid"
]

timestamp_columns = [
    "starttime", "endtime", "storetime"
]

string_columns = [
    "valueuom", "location", "locationcategory",
    "ordercategoryname", "secondaryordercategoryname",
    "ordercategorydescription", "patientweight",
    "statusdescription", "comments_date",
    "comments_canceledby", "comments"
]

for column in long_columns:
    if column in silver_df.columns:
        silver_df = silver_df.withColumn(
            column, F.col(column).cast("long")
        )

for column in timestamp_columns:
    if column in silver_df.columns:
        silver_df = silver_df.withColumn(
            column, F.col(column).cast("timestamp")
        )

for column in string_columns:
    if column in silver_df.columns:
        if dict(silver_df.dtypes)[column] == "string":
            silver_df = silver_df.withColumn(
                column, F.trim(F.col(column))
            )

# DQ flag: reversed interval (definitional check; source values are not modified)
if {"starttime", "endtime"}.issubset(silver_df.columns):
    silver_df = silver_df.withColumn(
        "dq_invalid_procedure_time",
        F.coalesce(
            F.col("starttime").isNotNull()
            & F.col("endtime").isNotNull()
            & (F.col("endtime") < F.col("starttime")),
            F.lit(False),
        ),
    )
else:
    silver_df = silver_df.withColumn("dq_invalid_procedure_time", F.lit(False))

silver_df.printSchema()
display(silver_df.limit(20))

# Expected published contract: Bronze columns/types in order (casts above applied), then the DQ flag
EXPECTED_SCHEMA = {f.name: f.dataType.simpleString() for f in silver_df.schema.fields}
assert list(EXPECTED_SCHEMA)[-1] == "dq_invalid_procedure_time", "DQ flag must be the last column"
assert EXPECTED_SCHEMA["dq_invalid_procedure_time"] == "boolean", "DQ flag must be boolean"

root
 |-- subject_id: long (nullable = true)
 |-- hadm_id: long (nullable = true)
 |-- stay_id: long (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: long (nullable = true)
 |-- value: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- location: string (nullable = true)
 |-- locationcategory: string (nullable = true)
 |-- orderid: long (nullable = true)
 |-- linkorderid: long (nullable = true)
 |-- ordercategoryname: string (nullable = true)
 |-- ordercategorydescription: string (nullable = true)
 |-- patientweight: double (nullable = true)
 |-- isopenbag: integer (nullable = true)
 |-- continueinnextdept: integer (nullable = true)
 |-- statusdescription: string (nullable = true)
 |-- originalamount: double (nullable = true)
 |-- originalrate: integer (nullable = true)
 |-- dq_invalid_procedure_time: boolean (nul

subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,value,valueuom,location,locationcategory,orderid,linkorderid,ordercategoryname,ordercategorydescription,patientweight,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate,dq_invalid_procedure_time
10000032,29079034,39553978,18704,2180-07-23T14:43:00.000Z,2180-07-23T14:44:00.000Z,2180-07-23T14:43:00.000Z,225966,1.0,None,null,null,3050203,3050203,Procedures,Task,39.4,0,0,FinishedRunning,1.0,0,false
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224275,566.0,min,null,null,769190,769190,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1,false
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224277,566.0,min,null,null,2668055,2668055,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1,false
10000690,25860671,37081114,17393,2150-11-02T20:00:00.000Z,2150-11-03T13:00:00.000Z,2150-11-03T14:50:00.000Z,224275,1020.0,min,RL Ant Forearm Medial,Peripheral - old,301419,301419,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,1020.0,1,false
10000690,25860671,37081114,17393,2150-11-02T20:38:00.000Z,2150-11-03T11:59:00.000Z,2150-11-03T12:34:00.000Z,224277,921.0,min,L Antecube,Peripheral - old,5207974,5207974,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,921.0,1,false
10000690,25860671,37081114,17393,2150-11-03T04:00:00.000Z,2150-11-03T04:01:00.000Z,2150-11-03T04:33:00.000Z,225459,1.0,None,null,null,8832126,8832126,Imaging,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T12:00:00.000Z,2150-11-05T12:01:00.000Z,2150-11-05T12:58:00.000Z,225402,1.0,None,null,null,4411905,4411905,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225401,1.0,None,null,null,7931107,7931107,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225814,1.0,None,null,null,2290746,2290746,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,null,2150-11-03T14:00:00.000Z,2150-11-06T17:03:00.000Z,2150-11-06T17:03:19.813Z,224277,4503.0,min,null,null,4940037,4940037,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,4503.0,1,false


In [0]:
# Report (profiling only) whether orderid is unique; uniqueness is not established by the documentation
if "orderid" in silver_df.columns:
    order_profile = silver_df.select(
        F.count("*").alias("rows"),
        F.sum(F.col("orderid").isNull().cast("int")).alias("null_orderid"),
        F.countDistinct("orderid").alias("distinct_orderid"),
    ).first()
    print("orderid profile [profiling only]:", order_profile.asDict())

orderid profile [profiling only]: {'rows': 808706, 'null_orderid': 0, 'distinct_orderid': 808706}


In [0]:
# Report exact duplicate procedure event records without removing them or blocking publication
# One aggregation pass over a single groupBy (no exceptAll); duplicates are preserved.
duplicate_summary = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("surplus_rows"),
    )
    .first()
)

duplicate_count = duplicate_summary["duplicate_groups"]
duplicate_surplus_rows = duplicate_summary["surplus_rows"]

print("Exact duplicate procedure event groups [preserved, report only]:", duplicate_count)
print("Rows beyond the first in those groups [preserved, report only]:", duplicate_surplus_rows)

Exact duplicate procedure event groups [preserved, report only]: 0
Rows beyond the first in those groups [preserved, report only]: 0


In [0]:
# Inspect NULL values in important event identifiers and timestamps
critical_columns = [
    "subject_id", "hadm_id", "stay_id",
    "itemid", "starttime", "endtime"
]

available_critical_columns = [
    c for c in critical_columns if c in silver_df.columns
]

quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in available_critical_columns
    ]
)

display(quality_df)

null_subject_id,null_hadm_id,null_stay_id,null_itemid,null_starttime,null_endtime
0,0,0,0,0,0


In [0]:
# Count procedure events with end times before start times (flagged, preserved, not corrected)
flag_summary = silver_df.select(
    F.sum(F.col("dq_invalid_procedure_time").cast("int")).alias("flagged"),
    F.sum(F.col("dq_invalid_procedure_time").isNull().cast("int")).alias("null_flags"),
).first()
invalid_time_count = flag_summary["flagged"] or 0

print("Events with endtime before starttime (flagged, preserved):", invalid_time_count)
assert flag_summary["null_flags"] == 0, "dq_invalid_procedure_time must never be NULL"

Events with endtime before starttime (flagged, preserved): 0


In [0]:
# Confirm all Bronze records are preserved by row count
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

Bronze rows: 808706
Silver rows: 808706


In [0]:
# Report NULL identifiers (review only; no NOT NULL is documented) and validate record preservation
quality = quality_df.first().asDict()

review_findings = {k: v for k, v in quality.items() if v != 0}
print("NULL findings [preserved, review only]:", review_findings or "none")

assert bronze_count == silver_count, "Record counts do not match"

print("Row-preservation check passed.")
print("Exact duplicate groups (preserved, review only):", duplicate_count)
print("Reversed time intervals (flagged, preserved):", invalid_time_count)

NULL findings [preserved, review only]: none
Row-preservation check passed.
Exact duplicate groups (preserved, review only): 0
Reversed time intervals (flagged, preserved): 0


In [0]:
# Publish the validated Silver procedure events table (CREATE OR REPLACE; downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.procedureevents


In [0]:
# Verify published procedure events and table structure
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

# DQ flag must be non-null in the published table; flagged count must match the pre-publish count
published_flags = published_df.select(
    F.sum(F.col("dq_invalid_procedure_time").isNull().cast("int")).alias("null_flags"),
    F.sum(F.col("dq_invalid_procedure_time").cast("int")).alias("flagged"),
).first()
assert published_flags["null_flags"] == 0, "Published dq_invalid_procedure_time contains NULLs"
assert (published_flags["flagged"] or 0) == invalid_time_count, "Published flag count differs from pre-publish count"

display(published_df.limit(20))

print("Silver procedureevents table verified successfully.")

Bronze rows: 808706
Silver rows: 808706
Published rows: 808706


subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,value,valueuom,location,locationcategory,orderid,linkorderid,ordercategoryname,ordercategorydescription,patientweight,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate,dq_invalid_procedure_time
10000032,29079034,39553978,18704,2180-07-23T14:43:00.000Z,2180-07-23T14:44:00.000Z,2180-07-23T14:43:00.000Z,225966,1.0,None,null,null,3050203,3050203,Procedures,Task,39.4,0,0,FinishedRunning,1.0,0,false
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224275,566.0,min,null,null,769190,769190,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1,false
10000032,29079034,39553978,null,2180-07-23T14:24:00.000Z,2180-07-23T23:50:00.000Z,2180-07-23T23:50:49.983Z,224277,566.0,min,null,null,2668055,2668055,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1,false
10000690,25860671,37081114,17393,2150-11-02T20:00:00.000Z,2150-11-03T13:00:00.000Z,2150-11-03T14:50:00.000Z,224275,1020.0,min,RL Ant Forearm Medial,Peripheral - old,301419,301419,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,1020.0,1,false
10000690,25860671,37081114,17393,2150-11-02T20:38:00.000Z,2150-11-03T11:59:00.000Z,2150-11-03T12:34:00.000Z,224277,921.0,min,L Antecube,Peripheral - old,5207974,5207974,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,921.0,1,false
10000690,25860671,37081114,17393,2150-11-03T04:00:00.000Z,2150-11-03T04:01:00.000Z,2150-11-03T04:33:00.000Z,225459,1.0,None,null,null,8832126,8832126,Imaging,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T12:00:00.000Z,2150-11-05T12:01:00.000Z,2150-11-05T12:58:00.000Z,225402,1.0,None,null,null,4411905,4411905,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225401,1.0,None,null,null,7931107,7931107,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,33125,2150-11-05T17:21:00.000Z,2150-11-05T17:22:00.000Z,2150-11-05T17:21:00.000Z,225814,1.0,None,null,null,2290746,2290746,Procedures,Task,55.3,0,0,FinishedRunning,1.0,0,false
10000690,25860671,37081114,null,2150-11-03T14:00:00.000Z,2150-11-06T17:03:00.000Z,2150-11-06T17:03:19.813Z,224277,4503.0,min,null,null,4940037,4940037,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,4503.0,1,false


Silver procedureevents table verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")